In [1]:
print("Hello World")

Hello World


In [5]:
from fp_ingest import load_qa_kb
import json

documents = load_qa_kb()

# Print the first record in a readable format
print(json.dumps(documents[0], indent=2))

{
  "qa_id": "qa_001",
  "doc_id": "doc_001",
  "question": "What is the dancing plague of 1518?",
  "answer": "The dancing plague of 1518, or dance epidemic of 1518, was a case of dancing mania that occurred in Strasbourg"
}


In [2]:
from dotenv import load_dotenv
load_dotenv()

from fp_ingest import load_qa_kb, build_index
from fp_rag_helper import RAGBase
from openai import OpenAI

# Fixed: Use the correct function name
documents = load_qa_kb()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

# Optional: Change this to a question about the Dancing Plague since that's your data
answer = assistant.rag("What caused the dancing plague of 1518?")
print(answer)

The cause was not known at the time. Modern theories suggest it may have been mass hysteria or mass psychogenic illness, or possibly ergot poisoning from toxic fungi in food.


In [3]:
def search(query):
    boost_dict = {"question": 3.0, "answer": 0.5}
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the knowledge base for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the knowledge base."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [4]:
instructions = """
You are a history study tutor. Answer questions about the Dancing Plague of 1518.

If you need information to answer the question, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches if needed. Try to expand your search by using new keywords based on the results you get.

At the end, ask if there are other areas the user wants to explore.
""".strip()

In [5]:
import json

question = "What caused the dancing plague of 1518?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
]

it = 1
while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            args = json.loads(item.arguments)
            if item.name == "search":
                result = search(**args)
            result_json = json.dumps(result, indent=2)
            messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": result_json,
            })
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            last_answer = item.content[0].text
            print(last_answer)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"Dancing Plague of 1518 cause what caused dancing plague of 1518 Strasbourg theories"}
iteration #2...
ASSISTANT:
The exact cause of the Dancing Plague of 1518 is still unknown. Modern historians and researchers usually point to two main theories:

- **Mass psychogenic illness / mass hysteria**: stress and social pressure may have triggered compulsive dancing in groups.
- **Ergot poisoning**: food contaminated with ergot fungi may have caused toxic or psychoactive effects.

So, the cause was never definitively established.  

Would you like to explore the theories, the events in Strasbourg, or how it eventually stopped?


In [11]:
import pandas as pd
df = pd.read_csv('feedback.csv')
df

,timestamp,question,answer,feedback,response_time
0,2026-08-01T04:40:18.713074,Who found out about the dancing plague?,The outbreak seems to have been first noticed ...,1,5.540724
1,2026-08-01T04:41:50.515928,What kind of dance did the affected people do?,The affected people did a kind of uncontrolled...,1,2.354920
2,2026-08-01T04:47:56.520444,Where there any famous people who got the danc...,Yes—there was at least one named person often ...,-1,3.943314
